# K-Means Customer Segmentation (Retail Bank) – Practice Skeleton

**Short name (GitHub):** `KMBank`

**Lab source:** Codecademy *K-Means Clustering* adapted to a retail-bank book (personas instead of handwritten digits).

Work this notebook first. Peek at `KMBank_Solution.ipynb` only when stuck. `KMBank.py` is a sklearn-shaped from-scratch `KMeans`.

**Files**
- `data/bank_customers.csv` — 2,000 customers × 10 features + planted `segment` (held out of `.fit`)
- `kmbank_flowchart.png` — desired outcome

**Not a credit decision, not a pricing engine, not a SAR.** Clustering groups similar rows. A human still names the bins and owns the treatment.


## Inline cheat-sheet (keep this cell visible)

See also **`KMBank_Cheatsheet.docx`**.

| Item | Formula / code |
|------|----------------|
| Sample vs feature | row = one customer; column = score, DTI, util, deposits, … |
| Scale first | \(x'=(x-\mu)/\sigma\) — score 575–785 must not dominate util 0–1 |
| Distance | \(d(x,c)=\sqrt{\sum_j(x'_j-c_j)^2}\) on the **scaled** row |
| Assign / update | nearest centroid; centroid = mean of its rows |
| Inertia | \(J=\sum_i\|x_i-c_{\ell_i}\|^2\) on the scaled matrix |
| Elbow | plot \(J(k)\); domain k=5 (five retail personas) |
| Purity / ARI | majority-map cluster → planted persona, then score |
| Inference | scale a new applicant with the **training** \(\mu,\sigma\), then `.predict` |

**Order:** scale → `fit` → map names → `predict` new rows with the same scaler.


## Desired outcome

![flowchart](kmbank_flowchart.png)

1. Load the book (samples × features). Drop `customer_id` and the persona label from `X`.
2. Scale columns. Credit score and utilization do not share a unit.
3. Choose \(k\) from the retail glossary (five personas) and confirm with an elbow.
4. Place centroids, assign, update, until the shift is below `tol`.
5. Profile each centroid in **original units** (the chart a branch manager can read).
6. Map cluster ids → persona names. Score only if a label exists.
7. Score four new applicants. Simulate k / n / noise / n_init.


## 0. Packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from sklearn.cluster import KMeans, MiniBatchKMeans
    from sklearn.preprocessing import StandardScaler
    from sklearn.decomposition import PCA
    from sklearn.metrics import adjusted_rand_score
    HAS_SK = True
except ImportError:
    HAS_SK = False
    print("sklearn missing — KMBank.KMeans + numpy scaler / PCA")

from KMBank import (
    KMeans as KMScratch,
    assign,
    inertia,
    map_clusters_to_truth,
    adjusted_rand,
    elbow_curve,
)

FEATS = [
    "credit_score", "income_k", "dti", "card_util", "deposit_k",
    "loan_k", "n_products", "tenure_mo", "late30_12m", "logins_30d",
]
PERSONAS = [
    "Prime Transactor",
    "Mass-Market Saver",
    "Card Revolver",
    "Stressed Book",
    "Affluent Relationship",
]
plt.rcParams["figure.figsize"] = (7, 4)
print("sklearn available:", HAS_SK)


## 1. Unsupervised clustering in a bank

The book is not labeled for this exercise. K-means answers two questions: how many groups (`k`), and what “similar” means (Euclidean distance to a centroid after z-scoring). Training = assign + update. Inference = nearest centroid. A planted persona column exists only so we can *grade* the clustering.

## 2. Load the retail book and scale it

2,000 customers, 400 in each of five personas. Features mix 300–850 scores with 0–1 utilization — scale before you cluster.

In [ ]:
# Task 2.1 — read data/bank_customers.csv
# X_raw = the 10 numeric columns in FEATS
# y     = segment (int 0-4)  — inspection only
# Do not put customer_id or segment_name into X.

book = X_raw = y = None  # YOUR CODE HERE
print(None if book is None else book.shape)


In [ ]:
# Task 2.2 — column-wise z-score. Keep mu, sd so new applicants use the same scaler.
# Do NOT scale with statistics computed on the row you are about to predict.

def scale_fit(X):
    # YOUR CODE HERE
    pass

def scale_apply(X, mu, sd):
    # YOUR CODE HERE
    pass


## 3. Look at the book before you cluster

In [ ]:
# Task 3.1 — scatter card_util vs credit_score, color by y.
# Task 3.2 — median score / util / lates / deposits by segment_name.
# Clustering will not see these colors.


## 4. From-scratch k-means on a 2-feature toy (DTI × util)

Banking analog of the Iris warm-up. Four functions: place, assign, update, loop.

In [ ]:
# Task 4 — from-scratch k-means on the 2-column matrix [dti, card_util] (scaled).
# random_centroids, assign_labels, update_centroids, kmeans_loop.
# Run k=3. Print last inertia.

def random_centroids(X, k, seed=0):
    # YOUR CODE HERE
    pass

def assign_labels(X, centroids):
    # YOUR CODE HERE
    pass

def update_centroids(X, labels, k):
    # YOUR CODE HERE
    pass

def kmeans_loop(X, k, max_iter=40, seed=0, tol=1e-4):
    # YOUR CODE HERE
    pass


## 5. Full 10-feature book — k=5

In [ ]:
# Task 5.1 — Why k=5 on this book?
# Task 5.2 — KMeans(n_clusters=5, n_init=10, random_state=0).fit(X)   # scaled X
# Print inertia_ and cluster sizes. Do not pass y.


## 6. Centroids as prototype customers

A centroid lives in the same 10-D space as a row, so it *is* a customer profile. Back-transform to dollars and percents.

In [ ]:
# Task 6.1 — C_raw = cluster_centers_ * sd + mu
# Task 6.2 — bar or table of score, util, deposits, lates, products per cluster.
# These bars are the banking analog of the 8×8 digit tiles.


## 7. Map cluster ids to persona names + score

In [ ]:
# Task 7.1 — majority vote: mode of y[labels==j] for each cluster j.
# Task 7.2 — purity and a crosstab of persona vs cluster.
# Optional: KMBank.map_clusters_to_truth(labels, y, k=5)


## 8. Elbow method

Inertia always drops with k. The retail glossary already says five personas; the elbow should be *compatible*, not a substitute for that story.

In [ ]:
# Task 8 — inertia for k=1..10 on scaled X. Mark k=5.
# Is the elbow exactly at 5, or just compatible with 5?


## 9. PCA view (10-D → 2-D)

In [ ]:
# Task 9 — 2-D view via SVD (or sklearn PCA). Color by y vs by labels_.
# Clustering ran in 10-D on scaled columns. PCA is a slide, not the model.


## 10. Alternate code that reaches the same idea

In [ ]:
# Task 10 — alternates
# A. init="random" vs "k-means++"
# B. fit on X_raw (no scale) and compare purity — why does it move?
# C. fit_predict
# D. MiniBatchKMeans if sklearn is installed


## 11. More practice

In [ ]:
# More practice
# P1. Cluster on score + dti + util + late30 only. Purity vs all 10 columns?
# P2. k=3 and k=8 on the full scaled book.
# P3. Only utilization + deposits. Which personas survive?


## 12. Four new applicants

In [ ]:
# Task 12 — four new applicants (raw units).
# Scale with the TRAINING mu, sd. Predict. Translate with the mapping from Task 7.
# Applicant D is messy on purpose.

new_raw = np.array([
    [790, 102, 0.20, 0.10, 30,  8, 3,  80, 0, 16],
    [580,  28, 0.55, 0.90,  1, 22, 2,  20, 5,  4],
    [760, 170, 0.30, 0.16, 90, 250, 6, 110, 0, 12],
    [640,  44, 0.38, 0.55,  8, 20, 3,  40, 2, 18],
], dtype=float)
# YOUR CODE HERE


## 13. Simulation — turn the knobs

In [ ]:
# Simulation knobs: k, n (subsample), noise σ on scaled features, n_init.
# Write run_once, run a small grid, plot inertia and purity vs k.

def run_once(k=5, n=2000, noise=0.0, n_init=10, seed=0):
    # YOUR CODE HERE
    pass


## Audience rewrite (Jočys checklist + McMurrey types)

| Audience | What they need | One sentence they should hear |
|----------|----------------|-------------------------------|
| Expert (Credit Risk / quant) | scaled Euclidean, inertia 4,670, ARI 0.945, unscaled ablation | k=5 on z-scored 10 columns recovers the planted book at purity 0.978; skip the scaler and score in the 700s swallows DTI. |
| Technician (CRM / campaign ops) | centroid table in dollars and percents | Five prototype customers. Drop a new row on the nearest prototype; treat the bin, not the cluster id. |
| Executive (Retail / CRO) | what changes in the branch | A first-pass grouping so offers, collections intensity, and relationship coverage are not one-size-fits-all. Not a scorecard. |
| Nonspecialist | no jargon | The bank piles customers who look alike on deposits, card use, and late payments, then names each pile. |

Data literacy: Risk gets the elbow; the branch gets the six-bar prototype chart.

Subject knowledge: do not define DTI or utilization for a credit officer. Do define that cluster `2` is a bin number until the glossary names it “Stressed Book.”

**What not to say**
- “The model declined the loan.” There is no decision threshold.
- “97% accurate like a PD model.” Labels were planted so we could *check* a clustering.
- “k=5 is proven by inertia.” Inertia keeps falling; five is the retail story plus a compatible bend.


## What this model can and cannot do

**Can**
- Group a numeric retail book into compact spherical personas.
- Hand a relationship manager a prototype row in original units.
- Assign a *new* scaled 10-vector to the nearest prototype in milliseconds.
- Warm up campaign design, limit-increase waves, or collections intensity tiers.

**Cannot**
- Replace a bureau scorecard or an IFRS-9 PD.
- Respect compliance rules (fair-lending, adverse action) — a cluster is not a reason code.
- Handle mixed categorical products without encoding.
- Stay stable if you skip the scaler or change column order at inference.

**Top banking applications of the same idea**
1. Retail persona / next-best-offer seeds
2. Card-util vs transactor bins
3. Deposit-heavy vs credit-heavy relationship groups
4. Collections early-stage vs late-stage intensity
5. Branch / region clustering on mix and NPL
6. Small-business cash-flow pattern groups
7. ATM / digital-channel usage clusters
8. Wealth-book share-of-wallet prototypes
9. Alert-queue grouping before a case is opened
10. Color-quantization analog: tier a limit grid

**Anti-applications**
- Approve / decline
- Pricing a revolving APR
- Filing a Suspicious Activity Report
- Anything that needs a probability and a hold-out KS


## Next steps

- Add a categorical product flag (one-hot) and re-scale.
- Try \(k=4\) after merging Prime and Affluent for a cheaper campaign grid.
- GMM if you need soft membership (“60% Revolver / 40% Stressed”).
- Reusable template: `KMBank_Reusable_Template.ipynb`.
- Sister lab on glyphs: `KMDigits`.
